# import libraries 

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,Lasso,Ridge,ElasticNet,SGDRegressor,HuberRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
import pickle

# load dataset

In [2]:
dataset=pd.read_csv('datasets/USA_Housing.csv')

In [3]:
dataset.head(2)

,Avg. Area Income,Avg. Area House Age,Avg. Area Number of Rooms,Avg. Area Number of Bedrooms,Area Population,Price,Address
0,79545.458574,5.682861,7.009188,4.09,23086.800503,1.059034e+06,"208 Michael Ferry Apt. 674\nLaurabury, NE 3701..."
1,79248.642455,6.002900,6.730821,3.09,40173.072174,1.505891e+06,"188 Johnson Views Suite 079\nLake Kathleen, CA..."


In [4]:
x=dataset.drop(['Address','Price'],axis=1)
y=dataset['Price']

# split data

In [5]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=0)

# Load the models 

In [8]:
models={
    'LinearRegression':LinearRegression(),
    'Lasso':Lasso(),
    'Ridge':Ridge(),
    'SVR':SVR(),
    'KNN':KNeighborsRegressor(),
    'RandomForestRegressor':RandomForestRegressor(),
    'HuberRegressor':HuberRegressor(),
    'RandomForestRegressor':RandomForestRegressor(),
    'Lightgb':lgb.LGBMRegressor(),
    'xgboost':xgb.XGBRFRegressor(),
    'SGDRegressor':SGDRegressor(),
    'ANN':MLPRegressor(hidden_layer_sizes=(100,),max_iter=100),
    'PolynomialRegressor':Pipeline([
        ('polly',PolynomialFeatures(degree=4)),
        ('linear',LinearRegression())
    ])
}

results=[]


#  Train and evalute models

In [10]:
import warnings
warnings.filterwarnings('ignore')
for name,model in models.items():
    model.fit(x_train,y_train)
    y_pred=model.predict(x_test)
    mae=mean_absolute_error(y_test,y_pred)
    mse=mean_squared_error(y_test,y_pred)
    rmmse=np.sqrt(mse)
    r2=r2_score(y_test,y_pred)
    

    results.append({
        'model':name,
        'mae':mae,
        'mse':mse,
        'rmse':rmmse,
        'r2':r2

    })

    with open (f'{name}.pkl','wb') as f:
        pickle.dump(model,f)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000108 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1256
[LightGBM] [Info] Number of data points in the train set: 4000, number of used features: 5
[LightGBM] [Info] Start training from score 1231911.452183


# model evalution results

In [11]:
results_df=pd.DataFrame(results)
results_df.to_csv('model_evalution_result.csv',index=False)